# MLQuantFineco - ML-Based Quantitative Trading Framework

This notebook demonstrates a complete pipeline for:
1. **Data Fetching** - Download stock/index data from yfinance or akshare
2. **Feature Engineering** - Generate 60+ technical indicators
3. **Model Training** - Train multiple ML models (Random Forest, XGBoost, LightGBM, SVM, LSTM)
4. **Model Comparison** - Compare model performance side-by-side
5. **Backtesting** - Simulate trading with transaction costs, compare against benchmark
6. **Visualization** - Equity curves, drawdowns, feature importance

## 0. Installation & Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Install dependencies if needed
# !pip install yfinance akshare scikit-learn xgboost lightgbm tensorflow matplotlib seaborn pandas numpy tabulate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
plt.style.use('seaborn-v0_8-whitegrid')

from ml_quant_fineco.data.fetcher import DataFetcher
from ml_quant_fineco.features.technical import FeatureEngineer, train_test_split_time
from ml_quant_fineco.models.registry import create_model, list_models
from ml_quant_fineco.backtest.engine import BacktestEngine
from ml_quant_fineco.utils.visualization import (
    plot_equity_curve, plot_drawdown, plot_monthly_returns,
    plot_feature_importance, plot_model_comparison, plot_signals
)

print("Available models:", list_models())

## 1. Configuration

Change the parameters below to analyze any stock or index.

In [ ]:
# ========== CONFIGURATION ==========
# Symbol: stock ticker (e.g. 'AAPL', 'MSFT', 'TSLA') or index name
SYMBOL = "AAPL"

# Data source: 'yfinance' for US/global markets, 'akshare' for Chinese A-shares
DATA_SOURCE = "yfinance"

# Is this an index? Set True for indices like 'S&P500', 'NASDAQ', 'CSI300'
IS_INDEX = False

# Date range (leave as None to use 'period')
START_DATE = "2018-01-01"
END_DATE = None  # None = up to today
PERIOD = "5y"     # Used if START_DATE is None

# Label method: 'return_sign' (classification) or 'return' (regression)
LABEL_METHOD = "return_sign"
FORWARD_PERIOD = 1  # predict 1 day ahead

# Test set ratio
TEST_RATIO = 0.2

# Models to train (choose from: rf, xgboost, lightgbm, svm, lstm)
MODELS_TO_TRAIN = ["rf", "xgboost", "lightgbm", "svm"]

# Backtest settings
INITIAL_CAPITAL = 100000
COMMISSION_RATE = 0.001   # 0.1% commission
STRATEGY = "long_only"     # 'long_only' or 'long_short'

print(f"Target: {SYMBOL} | Source: {DATA_SOURCE} | Index: {IS_INDEX}")
print(f"Models: {MODELS_TO_TRAIN}")

## 2. Data Fetching

In [ ]:
fetcher = DataFetcher(source=DATA_SOURCE)

raw_data = fetcher.fetch(
    symbol=SYMBOL,
    start_date=START_DATE,
    end_date=END_DATE,
    period=PERIOD,
    is_index=IS_INDEX,
)

print(f"Data shape: {raw_data.shape}")
print(f"Date range: {raw_data.index[0].date()} to {raw_data.index[-1].date()}")
raw_data.head()

In [ ]:
# Quick price chart
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(raw_data.index, raw_data["Close"])
ax.set_title(f"{SYMBOL} - Closing Price", fontsize=14, fontweight='bold')
ax.set_ylabel("Price")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/price_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Price chart saved to results/price_chart.png")

## 3. Feature Engineering

In [ ]:
fe = FeatureEngineer(
    label_method=LABEL_METHOD,
    forward_period=FORWARD_PERIOD,
)

features_df = fe.build(raw_data)
feature_cols = fe.get_feature_columns(features_df)

print(f"Total features: {len(feature_cols)}")
print(f"Total samples: {len(features_df)}")
print(f"\nFeature columns ({len(feature_cols)}):")
for i in range(0, len(feature_cols), 6):
    print("  ", ", ".join(feature_cols[i:i+6]))

In [ ]:
# Target distribution
print("Target distribution:")
print(features_df["target"].value_counts())
print(f"\nClass balance: {features_df['target'].mean():.2%} positive")

## 4. Train/Test Split

In [ ]:
train_df, test_df = train_test_split_time(features_df, test_ratio=TEST_RATIO)

X_train = train_df[feature_cols]
y_train = train_df["target"]
X_test = test_df[feature_cols]
y_test = test_df["target"]

print(f"Train: {X_train.shape[0]} samples ({X_train.index[0].date()} to {X_train.index[-1].date()})")
print(f"Test:  {X_test.shape[0]} samples ({X_test.index[0].date()} to {X_test.index[-1].date()})")

## 5. Model Training & Evaluation

In [ ]:
results = {}

for model_name in MODELS_TO_TRAIN:
    print(f"\n{'='*60}")
    print(f"Training: {model_name.upper()}")
    print(f"{'='*60}")

    # Determine task type
    task = "classification" if LABEL_METHOD == "return_sign" else "regression"

    # Special handling for LSTM (needs sequence_length param)
    if model_name == "lstm":
        model = create_model(model_name, task=task, sequence_length=20)
    else:
        model = create_model(model_name, task=task)

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    metrics = model.evaluate(X_test, y_test, task=task)
    results[model_name] = {
        "model": model,
        "metrics": metrics,
    }

    # Print metrics
    print(f"\n{model_name.upper()} Metrics:")
    for k, v in metrics.items():
        print(f"  {k:>12s}: {v:.4f}")

    # Feature importance (for tree-based models)
    if hasattr(model, 'feature_importance'):
        imp = model.feature_importance()
        imp.index = feature_cols
        fig = plot_feature_importance(
            imp, top_n=15,
            title=f"{model_name.upper()} - Top 15 Features"
        )
        plt.savefig(f'results/feature_importance_{model_name}.png', dpi=150, bbox_inches='tight')
        plt.show()

## 6. Model Comparison

In [ ]:
# Build comparison table
comparison_data = {}
for name, res in results.items():
    comparison_data[name.upper()] = res["metrics"]

comparison_df = pd.DataFrame(comparison_data).T
comparison_df.index.name = "Model"

print("\nModel Performance Comparison:")
print("="*70)
print(comparison_df.to_string())

# Bar chart comparison
task = "classification" if LABEL_METHOD == "return_sign" else "regression"
if task == "classification":
    plot_metrics = ["accuracy", "precision", "recall", "f1", "auc"]
else:
    plot_metrics = ["mse", "rmse", "mae"]

fig = plot_model_comparison(comparison_df, metrics=plot_metrics)
plt.savefig('results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Model comparison chart saved to results/model_comparison.png")

## 7. Backtesting

Run backtests for each model and compare against buy-and-hold benchmark.

In [ ]:
backtest_results = {}
backtest_metrics = {}

# Test prices (aligned with test set)
test_prices = test_df["Close"]

for name, res in results.items():
    model = res["model"]
    print(f"\nBacktesting {name.upper()}...")

    # Generate signals on test set
    if name == "lstm":
        predictions = model.predict(X_test)
        # LSTM predictions are shorter due to sequence window
        # We need to align with test prices
        seq_len = model.sequence_length
        aligned_prices = test_prices.iloc[seq_len:]
        aligned_signals = pd.Series(predictions, index=aligned_prices.index[:len(predictions)])
    else:
        predictions = model.predict(X_test)
        aligned_signals = pd.Series(predictions, index=X_test.index)
        aligned_prices = test_prices

    # Run backtest
    engine = BacktestEngine(
        initial_capital=INITIAL_CAPITAL,
        commission_rate=COMMISSION_RATE,
        strategy=STRATEGY,
    )

    bt_result = engine.run(aligned_prices, aligned_signals)
    bt_metrics = engine.compute_metrics()

    backtest_results[name] = bt_result
    backtest_metrics[name] = bt_metrics

    # Print key metrics
    print(f"  Total Return:     {bt_metrics['total_return']:>8.2%}")
    print(f"  Benchmark Return: {bt_metrics['benchmark_return']:>8.2%}")
    print(f"  Sharpe Ratio:     {bt_metrics['sharpe_ratio']:>8.3f}")
    print(f"  Max Drawdown:     {bt_metrics['max_drawdown']:>8.2%}")

## 8. Backtest Visualization

In [ ]:
# Find the best model by Sharpe ratio
best_model_name = max(backtest_metrics, key=lambda k: backtest_metrics[k]["sharpe_ratio"])
best_bt = backtest_results[best_model_name]

print(f"Best model by Sharpe ratio: {best_model_name.upper()}")

# Equity curve
fig = plot_equity_curve(
    best_bt,
    title=f"{SYMBOL} - {best_model_name.upper()} vs Buy & Hold",
)
plt.savefig('results/equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Drawdown chart
fig = plot_drawdown(
    best_bt,
    title=f"{SYMBOL} - {best_model_name.upper()} Drawdown",
)
plt.savefig('results/drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Trading signals
fig = plot_signals(
    best_bt["price"],
    best_bt["signal"],
    title=f"{SYMBOL} - {best_model_name.upper()} Trading Signals",
)
plt.savefig('results/trading_signals.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Comprehensive Benchmark Report

In [ ]:
# Build comprehensive backtest comparison table
report_rows = []
for name in backtest_metrics:
    m = backtest_metrics[name]
    report_rows.append({
        "Model": name.upper(),
        "Total Return": f"{m['total_return']:.2%}",
        "Annual Return": f"{m['annualized_return']:.2%}",
        "Sharpe": f"{m['sharpe_ratio']:.3f}",
        "Sortino": f"{m['sortino_ratio']:.3f}",
        "Max DD": f"{m['max_drawdown']:.2%}",
        "Win Rate": f"{m['win_rate']:.2%}",
        "Alpha": f"{m['alpha']:.2%}",
        "Beta": f"{m['beta']:.3f}",
        "Info Ratio": f"{m['information_ratio']:.3f}",
    })

# Add benchmark row
bm = list(backtest_metrics.values())[0]
report_rows.append({
    "Model": "BENCHMARK (Buy&Hold)",
    "Total Return": f"{bm['benchmark_return']:.2%}",
    "Annual Return": f"{bm['annualized_benchmark']:.2%}",
    "Sharpe": "-",
    "Sortino": "-",
    "Max DD": f"{bm['benchmark_max_drawdown']:.2%}",
    "Win Rate": "-",
    "Alpha": "-",
    "Beta": "1.000",
    "Info Ratio": "-",
})

report_df = pd.DataFrame(report_rows)
print("\n" + "="*100)
print(f"BACKTEST REPORT: {SYMBOL}")
print("="*100)
print(report_df.to_string(index=False))
print("="*100)

In [ ]:
# Equity curves for ALL models overlaid
fig, ax = plt.subplots(figsize=(14, 7))

for name, bt in backtest_results.items():
    equity = bt["portfolio_value"] / bt["portfolio_value"].iloc[0]
    ax.plot(equity.index, equity.values, label=f"{name.upper()} Strategy", linewidth=1.3)

# Benchmark
bench = list(backtest_results.values())[0]
bench_equity = bench["price"] / bench["price"].iloc[0]
ax.plot(bench_equity.index, bench_equity.values,
        label="Buy & Hold", color="black", linewidth=2, linestyle="--", alpha=0.7)

ax.set_title(f"{SYMBOL} - All Models Equity Curve Comparison", fontsize=14, fontweight='bold')
ax.set_ylabel("Normalized Value")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/all_models_equity.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Quick-Start Examples for Other Markets

Below are ready-to-use configurations for different markets.

In [ ]:
# === US Stock Example ===
# SYMBOL = "TSLA"
# DATA_SOURCE = "yfinance"
# IS_INDEX = False

# === US Index Example ===
# SYMBOL = "S&P500"  # or "NASDAQ", "DOW_JONES", "VIX"
# DATA_SOURCE = "yfinance"
# IS_INDEX = True

# === Chinese Stock Example ===
# SYMBOL = "600519"   # Kweichow Moutai
# DATA_SOURCE = "akshare"
# IS_INDEX = False

# === Chinese Index Example ===
# SYMBOL = "CSI300"   # or "SSE", "SZSE", "GEM"
# DATA_SOURCE = "akshare"
# IS_INDEX = True

print("Configuration examples shown above. Uncomment and modify the CONFIGURATION cell to use them.")

## 11. Summary

This notebook demonstrated the full MLQuantFineco pipeline:

| Component | Description |
|-----------|-------------|
| **Data Sources** | yfinance (US/global), akshare (China A-shares) |
| **Features** | 60+ technical indicators (MA, RSI, MACD, Bollinger, ATR, ADX, etc.) |
| **Models** | Random Forest, XGBoost, LightGBM, SVM, LSTM |
| **Backtest** | Transaction costs, slippage, long-only/long-short |
| **Benchmark** | Buy & Hold with full performance attribution |
| **Metrics** | Sharpe, Sortino, Calmar, Alpha, Beta, Max Drawdown, Information Ratio |

### Project Structure
```
ml_quant_fineco/
|-- data/fetcher.py         # Data fetching (yfinance, akshare)
|-- features/technical.py   # Feature engineering + train/test split
|-- models/
|   |-- base.py             # Base model interface
|   |-- sklearn_models.py   # RandomForest, SVM
|   |-- boosting_models.py  # XGBoost, LightGBM
|   |-- dl_models.py        # LSTM
|   |-- registry.py         # Model factory
|-- backtest/engine.py      # Backtesting engine
|-- utils/visualization.py  # Plotting utilities
main.ipynb                  # This notebook
results/                    # Output charts
```